In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
from extinctions import *
from preprocessing import *
from plotting_parameters import *

In [ ]:
bands_range = [520,900]
band_centers = np.arange(bands_range[0], bands_range[1]+1)
endmember_labels = ["oxCytc", "redCytc", "oxCytb", "redCytb", "oxCCO", "redCCO", "HHb", "HbO$_2$", "water", "lipids", "a'"]

extinction_dict = get_extinctions(bands_range)
cyt_c_ox = extinction_dict["cyt_c_ox"]
cyt_c_red = extinction_dict["cyt_c_red"]
cyt_b_ox = extinction_dict["cyt_b_ox"]
cyt_b_red = extinction_dict["cyt_b_red"]
cyt_oxi_ox = extinction_dict["cyt_oxi_ox"]
cyt_oxi_red = extinction_dict["cyt_oxi_red"]
hb = extinction_dict["hb"]
hbo2 = extinction_dict["hbo2"]
water = extinction_dict["water"]
fat = extinction_dict["fat"]
scatter_simple = (band_centers/500)**(-1.2)


M_scatter_orig = np.vstack((cyt_c_ox, cyt_c_red, cyt_b_ox, cyt_b_red, cyt_oxi_ox, cyt_oxi_red, hb, hbo2, water, fat, scatter_simple)).T
M_scatter_orig_smooth = smooth_spectral(M_scatter_orig.T,5).T


spectra_folder = "../mc_sim/spectra_mc/"

hb_mc = np.loadtxt(spectra_folder + "m_hhb_50.txt")
hbo2_mc = np.loadtxt(spectra_folder + "m_hbo2_50.txt")
cyt_c_ox_mc = np.loadtxt(spectra_folder + "m_cyt_c_ox_20.txt")
cyt_c_red_mc = np.loadtxt(spectra_folder + "m_cyt_c_red_20.txt")
cyt_b_ox_mc = np.loadtxt(spectra_folder + "m_cyt_b_ox_20.txt")
cyt_b_red_mc = np.loadtxt(spectra_folder + "m_cyt_b_red_20.txt")
cyt_oxi_ox_mc = np.loadtxt(spectra_folder + "m_cyt_oxi_ox_20.txt")
cyt_oxi_red_mc = np.loadtxt(spectra_folder + "m_cyt_oxi_red_20.txt")
scatter_mc = np.loadtxt(spectra_folder + "m_scatter_40.txt")
water_mc = np.loadtxt(spectra_folder + "m_water_200.txt")
fat_mc = np.loadtxt(spectra_folder + "m_fat_200.txt")

M_mc = np.vstack((cyt_c_ox_mc, cyt_c_red_mc, cyt_b_ox_mc, cyt_b_red_mc, cyt_oxi_ox_mc, cyt_oxi_red_mc, hb_mc, hbo2_mc, water_mc, fat_mc, scatter_mc)).T
M_mc_smooth = smooth_spectral(M_mc.T,5).T

M_hb = np.vstack((cyt_c_ox, cyt_c_red, cyt_b_ox, cyt_b_red, cyt_oxi_ox, cyt_oxi_red, hb_mc, hbo2_mc, water, fat, scatter_simple)).T
M_hb = smooth_spectral(M_hb.T,5).T

In [ ]:
def icem(spectr, t, lmda=0, labels=None, device='cpu', Rinv=None):
    '''
    Improved constrained energy minimization (ICEM) algorithm.
    input:
        spectr: image to unmix, shape (...,k) where k is the number of bands and ... are the spatial or time dimensions
        t: endmember to unmix, shape (k,)
        lmda: regularization parameter, float
    output:
        heatmap, np.array
    '''
    # ignore pixels that are not tissue
    spectr_tissue = spectr
    if labels is not None:
        spectr_tissue = spectr[labels != 4]

    was_numpy = isinstance(spectr, np.ndarray)
    spectr = torch.tensor(spectr, device=device).float()
    spectr_tissue = torch.tensor(spectr_tissue, device=device).float()
    t = torch.tensor(t, device=device).float()

    input_shape = spectr.shape
    M = spectr_tissue.reshape(-1, input_shape[-1])
    M_all = spectr.reshape(-1, input_shape[-1])

    N, p = M.shape
    if Rinv is None:
        R_hat = torch.mm(M.T, M) / N + lmda * torch.eye(p, device=device)
        Rinv = torch.inverse(R_hat)
    t_Rinv = torch.mv(Rinv.T, t)
    denom = torch.dot(t, t_Rinv)
    y = torch.mv(M_all, t_Rinv) / denom

    if was_numpy:
        y =  y.cpu().numpy()

    return y.reshape(input_shape[:-1]), Rinv

In [ ]:
def generate_heatmaps(absorbance, reference_spectrum, labels, folder):
    n = M_scatter_orig_smooth.shape[1]
    l, m = absorbance.shape[:2]
    diff_absorbance = absorbance - reference_spectrum[None,None,:]

    heatmaps_osp = np.zeros((l,m,n))
    heatmaps_osp_diff = np.zeros((l,m,n))
    heatmaps_osp_diff_mc = np.zeros((l,m,n))
    heatmaps_icem = np.zeros((l,m,n))
    heatmaps_icem_diff = np.zeros((l,m,n))
    heatmaps_icem_diff_mc = np.zeros((l,m,n))
    for i in range(n):
        endmembers_remove = np.delete(M_scatter_orig_smooth, i, axis=1).T
        endmember_target = M_scatter_orig_smooth[:,i]
        heatmaps_osp[:,:,i] = osp(absorbance, endmembers_remove, endmember_target)
        heatmaps_osp_diff[:,:,i] = osp(diff_absorbance, endmembers_remove, endmember_target)

        heatmaps_icem[:,:,i], Rinv_abs = icem(absorbance, endmember_target, 1, labels=labels)
        heatmaps_icem_diff[:,:,i], Rinv_diff_abs = icem(diff_absorbance, endmember_target, 1, labels=labels)


        endmembers_remove = np.delete(M_mc_smooth, i, axis=1).T
        endmember_target = M_mc_smooth[:,i]
        heatmaps_osp_diff_mc[:,:,i] = osp(diff_absorbance, endmembers_remove, endmember_target)
        heatmaps_icem_diff_mc[:,:,i], _ = icem(diff_absorbance, endmember_target, 1, labels=labels)

    if not os.path.exists(folder):
        os.makedirs(folder)
    np.save(folder + "heatmaps_osp.npy", heatmaps_osp)
    np.save(folder + "heatmaps_osp_diff.npy", heatmaps_osp_diff)
    np.save(folder + "heatmaps_osp_diff_mc.npy", heatmaps_osp_diff_mc)
    np.save(folder + "heatmaps_icem.npy", heatmaps_icem)
    np.save(folder + "heatmaps_icem_diff.npy", heatmaps_icem_diff)
    np.save(folder + "heatmaps_icem_diff_mc.npy", heatmaps_icem_diff_mc)
    np.save(folder + "labels.npy", labels)

    return heatmaps_osp, heatmaps_osp_diff, heatmaps_osp_diff_mc, heatmaps_icem, heatmaps_icem_diff, heatmaps_icem_diff_mc

### Own labels

In [ ]:
class_labels = ["Not labled", "Normal", "Tumor", "Hypervascularized", "Background"]
bands_range = [520,900]
band_centers = np.arange(bands_range[0], bands_range[1]+1)
# gradeIVpatients = ["008-01", "008-02", "010-03", "012-01", "012-02", "015-01", "016-04", "016-05", "017-01", "020-01", "039-01"]
patients = os.listdir("../../datasets/npj_database")

raw_data_all = []
preprocessed_data_all = []
labels_all = []
# labels_binary_all = []
heatmapts_osp_all = []
heatmapts_osp_diff_all = []
heatmapts_osp_diff_mc_all = []
heatmapts_icem_all = []
heatmapts_icem_diff_all = []
heatmapts_icem_diff_mc_all = []

for patient in patients:
    # data_folder = f"../../datasets/helicoid/{patient}"
    data_folder = f"../../datasets/npj_database/{patient}"
    # save_folder = f"./npj_database/{patient}/"
    # if not os.path.exists(save_folder):
    #     os.makedirs(save_folder)
    # print(patient)
        
    # # Load the sp data
    # img = sp.open_image(data_folder + "/raw.hdr")
    # white_ref = sp.open_image(data_folder + "/whiteReference.hdr")
    # dark_ref = sp.open_image(data_folder + "/darkReference.hdr")
    # labels = np.load(data_folder + "/labels.npy")
    # labels_binary = np.load(data_folder + "/labels_binary.npy")
    gt_map = sp.open_image(data_folder + "/gtMap.hdr")
    labels = gt_map.asarray().squeeze()

    # check if labels is 1 anywhere
    if np.sum(labels == 1) == 0:
        print(f"No normal tissue labeled for patient {patient}! Skipping...")
        continue

#     # np.save(save_folder + "labels.npy", labels)
#     # np.save(save_folder + "labels_binary.npy", labels_binary)
#     np.save(save_folder + "gtMap.npy", gt_map.asarray().squeeze())

#     # plot_img(img, labels, class_labels, class_colors=["white", tum_blue_dark_2, tum_orange, tum_red, tum_grey_1])

#     # preprocessing
#     img_interp, band_centers = bands_lin_interpolation(img, img.bands.centers, bands_range)
#     white_ref_interp, _ = bands_lin_interpolation(white_ref, img.bands.centers, bands_range)
#     dark_ref_interp, _ = bands_lin_interpolation(dark_ref, img.bands.centers, bands_range)
#     img_calib = calibrate_img(img_interp, white_ref_interp, dark_ref_interp)
#     img_calib_norm = img_calib / np.linalg.norm(img_calib, axis=(0,1), ord=1, keepdims=True)
#     img_smooth = smooth_spectral(img_calib_norm, 5)
#     absorbance = -np.log(img_smooth - np.min(img_smooth.flatten()) + 1e-8)
#     np.save(save_folder + "preprocessed.npy", absorbance)

#     # select random spectrum from normal class
#     np.random.seed(1)
#     normal_idx = np.argwhere(gt_map.asarray() == 1)
#     random_idx = np.random.choice(normal_idx.shape[0])
#     reference_spectrum = absorbance[normal_idx[random_idx,0], normal_idx[random_idx,1], :]
    

#     # heatmaps_osp, heatmaps_osp_diff, heatmaps_osp_diff_mc, heatmaps_icem, heatmaps_icem_diff, heatmaps_icem_diff_mc = generate_heatmaps(absorbance, reference_spectrum, labels, save_folder)
#     heatmaps_osp, heatmaps_osp_diff, heatmaps_osp_diff_mc, heatmaps_icem, heatmaps_icem_diff, heatmaps_icem_diff_mc = generate_heatmaps(absorbance, reference_spectrum, labels, save_folder)

#     filter = (labels != 0) & (labels != 4)
#     # filter = labels_binary != 0

#     # raw_data_all.append(img_interp[filter])
#     # preprocessed_data_all.append(absorbance[filter])
#     # # labels_all.append(labels[filter])
#     # # labels_binary_all.append(labels_binary[filter])
#     # heatmapts_osp_all.append(heatmaps_osp[filter])
#     # heatmapts_osp_diff_all.append(heatmaps_osp_diff[filter])
#     # heatmapts_osp_diff_mc_all.append(heatmaps_osp_diff_mc[filter])
#     # heatmapts_icem_all.append(heatmaps_icem[filter])
#     # heatmapts_icem_diff_all.append(heatmaps_icem_diff[filter])
#     # heatmapts_icem_diff_mc_all.append(heatmaps_icem_diff_mc[filter])

# # raw_data_all = np.vstack(raw_data_all)
# # preprocessed_data_all = np.vstack(preprocessed_data_all)
# # # labels_all = np.concatenate(labels_all)
# # # labels_binary_all = np.concatenate(labels_binary_all)
# # heatmapts_osp_all = np.vstack(heatmapts_osp_all)
# # heatmapts_osp_diff_all = np.vstack(heatmapts_osp_diff_all)
# # heatmapts_osp_diff_mc_all = np.vstack(heatmapts_osp_diff_mc_all)
# # heatmapts_icem_all = np.vstack(heatmapts_icem_all)
# # heatmapts_icem_diff_all = np.vstack(heatmapts_icem_diff_all)
# # heatmapts_icem_diff_mc_all = np.vstack(heatmapts_icem_diff_mc_all)

In [ ]:
folder = "./tumor_non_tumor/"
if not os.path.exists(folder):
    os.makedirs(folder)
np.save(folder + "raw_data_all.npy", raw_data_all)
np.save(folder + "preprocessed_data_all.npy", preprocessed_data_all)
np.save(folder + "labels_all.npy", labels_all)
np.save(folder + "labels_binary_all.npy", labels_binary_all)
np.save(folder + "heatmapts_osp_all.npy", heatmapts_osp_all)
np.save(folder + "heatmapts_osp_diff_all.npy", heatmapts_osp_diff_all)
np.save(folder + "heatmapts_osp_diff_mc_all.npy", heatmapts_osp_diff_mc_all)
np.save(folder + "heatmapts_icem_all.npy", heatmapts_icem_all)
np.save(folder + "heatmapts_icem_diff_all.npy", heatmapts_icem_diff_all)
np.save(folder + "heatmapts_icem_diff_mc_all.npy", heatmapts_icem_diff_mc_all)


# # # load the data
# class_labels = ["Not labled", "Normal", "Tumor", "Hypervascularized", "Background"]
# raw_data_all = np.load(folder + "raw_data_all.npy")
# preprocessed_data_all = np.load(folder + "preprocessed_data_all.npy")
# labels_all = np.load(folder + "labels_all.npy")
# heatmapts_osp_all = np.load(folder + "heatmapts_osp_all.npy")
# heatmapts_osp_diff_all = np.load(folder + "heatmapts_osp_diff_all.npy")
# heatmapts_osp_diff_mc_all = np.load(folder + "heatmapts_osp_diff_mc_all.npy")
# heatmapts_icem_all = np.load(folder + "heatmapts_icem_all.npy")
# heatmapts_icem_diff_all = np.load(folder + "heatmapts_icem_diff_all.npy")
# heatmapts_icem_diff_mc_all = np.load(folder + "heatmapts_icem_diff_mc_all.npy")

In [ ]:
# get class ratios

patients = os.listdir("./npj_database/")
sort = np.argsort(patients)
patients = np.array(patients)[sort]
print(patients)

labels_all = []
for patient in os.listdir("./npj_database/"):
    # check if gtMap.npy exists
    if not os.path.exists(f"./npj_database/{patient}/gtMap.npy"):
        # delete the folder
        os.rmdir(f"./npj_database/{patient}")
        continue
    labels = np.load(f"./npj_database/{patient}/gtMap.npy")
    labels_all.append(labels.flatten())
labels_all = np.concatenate(labels_all)


normal_sum = np.sum(labels_all == 1)
tumor_sum = np.sum(labels_all == 2)
blood_sum = np.sum(labels_all == 3)

normal_ratio = normal_sum / (normal_sum + tumor_sum + blood_sum)
tumor_ratio = tumor_sum / (normal_sum + tumor_sum + blood_sum)
blood_ratio = blood_sum / (normal_sum + tumor_sum + blood_sum)


print(f"Normal ratio: {normal_ratio:.2f}, Tumor ratio: {tumor_ratio:.2f}, Blood ratio: {blood_ratio:.2f}")

### T-SNE

In [ ]:
fig, ax = plot_tsne(raw_data_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("Raw data")
fig, ax = plot_tsne(preprocessed_data_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("Preprocessed data")
fig, ax = plot_tsne(heatmapts_osp_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("OSP")
fig, ax = plot_tsne(heatmapts_osp_diff_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("OSP diff")
fig, ax = plot_tsne(heatmapts_osp_diff_mc_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("OSP diff mc")
fig, ax = plot_tsne(heatmapts_icem_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("ICEM")
fig, ax = plot_tsne(heatmapts_icem_diff_all, labels_all, class_labels, figsize=(10,10), mode="all")
ax.set_title("ICEM diff")

In [ ]:
def clip_rescale(heatmaps, lower_clip=None, upper_clip=None, center=None):
    if lower_clip is None:
        lower_clip = np.percentile(heatmaps, 0.1, axis=0, keepdims=True)
    if upper_clip is None:
        upper_clip = np.percentile(heatmaps, 99.9, axis=0, keepdims=True)
    heatmaps = np.clip(heatmaps, lower_clip, upper_clip)
    heatmaps = heatmaps / (upper_clip - lower_clip)
    if center is None:
        center = np.percentile(heatmaps, 50, axis=0, keepdims=True)
    heatmaps = heatmaps - center + 0.5
    return heatmaps, lower_clip, upper_clip, center

def plot_map_dist(heatmaps, endmember_labels):
    n = heatmaps.shape[1]
    # plt.figure()
    # for i in range(n):
    #     plt.hist(heatmaps[:,i], bins=100, alpha=0.5, label=endmember_labels[i])
    #     plt.legend(endmember_labels, loc='center left', bbox_to_anchor=(1, 0.5))
    fig, axs = plt.subplots(1, n, figsize=(n*3, 3))
    for i in range(n):
        axs[i].hist(heatmaps[:,i], bins=100, alpha=0.5)
        axs[i].set_title(endmember_labels[i])

### CEM

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

pca = PCA(11)
pca_data = pca.fit_transform(preprocessed_data)
# pca_data, lower_clip_pca, upper_clip_pca, center_pca = clip_rescale(pca_data)
plot_map_dist(pca_data, [f"PCA component {i}" for i in range(1,12)])
np.save(folder + "pca_data.npy", pca_data)

In [ ]:
class_labels = ["Not labled", "Normal", "Tumor", "Hypervascularized", "Background"]
bands_range = [520,900]
band_centers = np.arange(bands_range[0], bands_range[1]+1)
gradeIVpatients = ["008-01", "008-02", "010-03", "012-01", "012-02", "015-01", "016-04", "016-05", "017-01", "020-01", "039-01"]

preprocessed_data_tissue = []
preprocessed_data_tissue_rel = []

for patient in gradeIVpatients:
    data_folder = f"../../datasets/helicoid/{patient}"
        
    # Load the sp data
    img = sp.open_image(data_folder + "/raw.hdr")
    white_ref = sp.open_image(data_folder + "/whiteReference.hdr")
    dark_ref = sp.open_image(data_folder + "/darkReference.hdr")
    gt_map = sp.open_image(data_folder + "/gtMap.hdr")

    labels = np.load(data_folder + "/labels.npy")

    # preprocessing
    img_interp, band_centers = bands_lin_interpolation(img, img.bands.centers, bands_range)
    white_ref_interp, _ = bands_lin_interpolation(white_ref, img.bands.centers, bands_range)
    dark_ref_interp, _ = bands_lin_interpolation(dark_ref, img.bands.centers, bands_range)
    img_calib = calibrate_img(img_interp, white_ref_interp, dark_ref_interp)
    img_calib_norm = img_calib / np.linalg.norm(img_calib, axis=(0,1), ord=1, keepdims=True)
    img_smooth = smooth_spectral(img_calib_norm, 5)
    absorbance = -np.log(img_smooth - np.min(img_smooth.flatten()) + 1e-8)

    filter = labels == 4

    preprocessed_data_tissue.append(absorbance[filter])

    np.random.seed(1)
    normal_idx = np.argwhere(gt_map.asarray() == 1)
    random_idx = np.random.choice(normal_idx.shape[0])
    reference_spectrum = absorbance[normal_idx[random_idx,0], normal_idx[random_idx,1], :]
    preprocessed_data_tissue_rel.append(absorbance[filter] - reference_spectrum[None,:])

preprocessed_data_tissue = np.vstack(preprocessed_data_tissue)
preprocessed_data_tissue_rel = np.vstack(preprocessed_data_tissue_rel)
print(preprocessed_data_tissue.shape)
print(preprocessed_data_tissue_rel.shape)

In [ ]:
M = torch.tensor(preprocessed_data_tissue)
N, k = M.shape
R_hat = torch.einsum('ij,ik->jk', M, M) / N + torch.eye(k)
Rinv_tisue = torch.inverse(R_hat)

M = torch.tensor(preprocessed_data_tissue_rel)
R_hat = torch.einsum('ij,ik->jk', M, M) / N + torch.eye(k)
Rinv_tisue_rel = torch.inverse(R_hat)

In [ ]:
for patient in gradeIVpatients:
    print(patient)
    data_folder = f"../../datasets/helicoid/{patient}"
    save_folder = f"./cem_global/{patient}/"
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
        
    # Load the sp data
    img = sp.open_image(data_folder + "/raw.hdr")
    white_ref = sp.open_image(data_folder + "/whiteReference.hdr")
    dark_ref = sp.open_image(data_folder + "/darkReference.hdr")
    labels = np.load(data_folder + "/labels.npy")
    labels_binary = np.load(data_folder + "/labels_binary.npy")
    gt_map = sp.open_image(data_folder + "/gtMap.hdr")

    np.save(save_folder + "labels.npy", labels)
    np.save(save_folder + "labels_binary.npy", labels_binary)
    np.save(save_folder + "gtMap.npy", gt_map.asarray().squeeze())

    plot_img(img, labels, class_labels, class_colors=["white", tum_blue_dark_2, tum_orange, tum_red, tum_grey_1])

    # preprocessing
    img_interp, band_centers = bands_lin_interpolation(img, img.bands.centers, bands_range)
    white_ref_interp, _ = bands_lin_interpolation(white_ref, img.bands.centers, bands_range)
    dark_ref_interp, _ = bands_lin_interpolation(dark_ref, img.bands.centers, bands_range)
    img_calib = calibrate_img(img_interp, white_ref_interp, dark_ref_interp)
    img_calib_norm = img_calib / np.linalg.norm(img_calib, axis=(0,1), ord=1, keepdims=True)
    img_smooth = smooth_spectral(img_calib_norm, 5)
    absorbance = -np.log(img_smooth - np.min(img_smooth.flatten()) + 1e-8)

    # select random spectrum from normal class
    np.random.seed(1)
    normal_idx = np.argwhere(gt_map.asarray() == 1)
    random_idx = np.random.choice(normal_idx.shape[0])
    reference_spectrum = absorbance[normal_idx[random_idx,0], normal_idx[random_idx,1], :]


    n = M_scatter_orig_smooth.shape[1]
    l, m = absorbance.shape[:2]
    diff_absorbance = absorbance - reference_spectrum[None,None,:]

    heatmaps_osp = np.zeros((l,m,n))
    heatmaps_osp_diff = np.zeros((l,m,n))
    heatmaps_osp_diff_mc = np.zeros((l,m,n))
    heatmaps_icem = np.zeros((l,m,n))
    heatmaps_icem_diff = np.zeros((l,m,n))
    heatmaps_icem_diff_mc = np.zeros((l,m,n))
    for i in range(n):
        endmembers_remove = np.delete(M_scatter_orig_smooth, i, axis=1).T
        endmember_target = M_scatter_orig_smooth[:,i]
        heatmaps_osp[:,:,i] = osp(absorbance, endmembers_remove, endmember_target)
        heatmaps_osp_diff[:,:,i] = osp(diff_absorbance, endmembers_remove, endmember_target)

        heatmaps_icem[:,:,i], Rinv_abs = icem(absorbance, endmember_target, 1, Rinv=Rinv_tisue)
        heatmaps_icem_diff[:,:,i], Rinv_diff_abs = icem(diff_absorbance, endmember_target, 1, Rinv=Rinv_tisue_rel)


        endmembers_remove = np.delete(M_mc_smooth, i, axis=1).T
        endmember_target = M_mc_smooth[:,i]
        heatmaps_osp_diff_mc[:,:,i] = osp(diff_absorbance, endmembers_remove, endmember_target)
        heatmaps_icem_diff_mc[:,:,i], _ = icem(diff_absorbance, endmember_target, 1, Rinv=Rinv_tisue_rel)

    np.save(save_folder + "heatmaps_osp.npy", heatmaps_osp)
    np.save(save_folder + "heatmaps_osp_diff.npy", heatmaps_osp_diff)
    np.save(save_folder + "heatmaps_osp_diff_mc.npy", heatmaps_osp_diff_mc)
    np.save(save_folder + "heatmaps_icem.npy", heatmaps_icem)
    np.save(save_folder + "heatmaps_icem_diff.npy", heatmaps_icem_diff)
    np.save(save_folder + "heatmaps_icem_diff_mc.npy", heatmaps_icem_diff_mc)